<div class="alert alert-block alert-success" style="font-family: Times New Roman">
    <h4><strong>Laboratory Task 2</strong></h4>
<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
    <b>Instruction:</b> Perform a single forward pass and compute for the error.
</p>

$$x = \begin{bmatrix} 1 \\ 0 \\ 1 \end{bmatrix}$$

$$y = \begin{bmatrix} 1 \end{bmatrix}$$

$$f=max(0, Z_n)$$

$$\text{hidden unit weights} =
\begin{bmatrix}
w_{11} = 0.2 && w_{12} = -0.3 \\
w_{13} = 0.4 && w_{14} = 0.1 \\
w_{15} = -0.5 && w_{16} = 0.2
\end{bmatrix}$$

$$\text{output unit weights} =
\begin{bmatrix}
w_{21} = -0.3 \\
w_{22} = -0.2
\end{bmatrix}$$

$$\theta =
\begin{bmatrix}
\theta_{1} = -0.4 \\
\theta_{2} = 0.2 \\
\theta_{3} = 0.1
\end{bmatrix}$$
</div>

### My Approach

Before writing any code, I want to be clear about **what network this problem is describing**, because that determines how I wire the equations together.

- **Input layer:** 3 features, $x = [x_1, x_2, x_3]^T = [1, 0, 1]^T$.
- **Hidden layer:** 2 units ($H_1$, $H_2$). Each hidden unit looks at *all three* inputs, so there are $3 \times 2 = 6$ hidden weights ($w_{11}$ … $w_{16}$) plus 2 biases ($\theta_1$, $\theta_2$) — one bias per hidden unit.
- **Output layer:** 1 unit ($\hat{y}$). It looks at the 2 hidden activations, so there are 2 output weights ($w_{21}$, $w_{22}$) plus 1 bias ($\theta_3$).
- **Activation function:** $f(Z) = \max(0, Z)$, i.e. **ReLU**, applied at *every* neuron (hidden **and** output), since the problem states it as the general activation $f$ for $Z_n$.

The indexing convention used in the given weight matrix is "row = hidden unit, column = input index pairing", e.g. $w_{11}, w_{13}, w_{15}$ all feed into $H_1$ (odd subscripts), while $w_{12}, w_{14}, w_{16}$ all feed into $H_2$ (even subscripts). This matches the pattern already introduced in the lecture material:

$$Z_1 = (w_{11} \cdot x_1) + (w_{13} \cdot x_2) + (w_{15} \cdot x_3) + \theta_1$$
$$Z_2 = (w_{12} \cdot x_1) + (w_{14} \cdot x_2) + (w_{16} \cdot x_3) + \theta_2$$

So my plan is:
1. Encode the given numbers as NumPy arrays.
2. Compute the hidden pre-activations $Z_1, Z_2$ by hand (dot product + bias) and confirm with code.
3. Apply ReLU to get $H_1, H_2$.
4. Compute the output pre-activation $Z_3$ and apply ReLU to get $\hat{y}$.
5. Compute the **error** between $\hat{y}$ and the target $y$, using a couple of common error definitions so the meaning of "error" is unambiguous.
6. Re-do the whole computation in **vectorized matrix form** (the way it is normally done in real code) as a sanity check.
7. Cross-check everything against PyTorch's own linear layers, so I'm confident the manual math and the NumPy code agree with a trusted deep learning library.

In [1]:
import numpy as np

# ---- Given data -------------------------------------------------------
x = np.array([1, 0, 1], dtype=float)      # input vector (3,)
y = np.array([1], dtype=float)            # target/actual output

# hidden layer weights, arranged so that column 0 -> H1, column 1 -> H2
# row i corresponds to input x_i
W_h = np.array([
    [0.2, -0.3],   # w11, w12  (multiply x1)
    [0.4,  0.1],   # w13, w14  (multiply x2)
    [-0.5, 0.2],   # w15, w16  (multiply x3)
])
theta_h = np.array([-0.4, 0.2])           # theta1 (H1 bias), theta2 (H2 bias)

# output layer weights
W_o = np.array([-0.3, -0.2])              # w21, w22
theta_o = 0.1                             # theta3 (output bias)

print("x       =", x)
print("y       =", y)
print("W_h =\n", W_h)
print("theta_h =", theta_h)
print("W_o     =", W_o)
print("theta_o =", theta_o)

x       = [1. 0. 1.]
y       = [1.]
W_h =
 [[ 0.2 -0.3]
 [ 0.4  0.1]
 [-0.5  0.2]]
theta_h = [-0.4  0.2]
W_o     = [-0.3 -0.2]
theta_o = 0.1


### Step 1 — Hidden layer weighted sum ($Z_1$, $Z_2$)

I compute these first "by hand" (writing out every term explicitly) to mirror exactly the formula from the notes, and then double-check with the dot-product form. If both agree, I know I indexed the weights correctly.

In [2]:
w11, w12 = W_h[0]
w13, w14 = W_h[1]
w15, w16 = W_h[2]
theta1, theta2 = theta_h
x1, x2, x3 = x

# --- explicit, term-by-term (matches the lecture formula) ---
Z1_manual = (w11 * x1) + (w13 * x2) + (w15 * x3) + theta1
Z2_manual = (w12 * x1) + (w14 * x2) + (w16 * x3) + theta2

# --- vectorized dot-product form (equivalent, less error-prone) ---
Z_hidden = x @ W_h + theta_h   # shape (2,) -> [Z1, Z2]

print(f"Z1 (manual)     = {Z1_manual}")
print(f"Z2 (manual)     = {Z2_manual}")
print(f"[Z1, Z2] (dot)  = {Z_hidden}")

assert np.allclose([Z1_manual, Z2_manual], Z_hidden), "Manual and vectorized Z do not match!"
print("\nManual computation matches the vectorized computation. Good to proceed.")

Z1 (manual)     = -0.7
Z2 (manual)     = 0.10000000000000003
[Z1, Z2] (dot)  = [-0.7  0.1]

Manual computation matches the vectorized computation. Good to proceed.


### Step 2 — Apply the activation function (ReLU) to get $H_1$, $H_2$

The problem defines $f = \max(0, Z_n)$, i.e. ReLU. Notice that $Z_1$ is **negative**. ReLU clips any negative pre-activation to exactly 0 — this is the well-known "dying ReLU" behavior. It doesn't mean anything went wrong; it's simply what ReLU does, and it will matter later in Lab 3 when we compute gradients (a unit whose output is clipped to 0 also has zero gradient flowing through it).

In [3]:
# ReLU activation: f(z) = max(0, z), applied element-wise.
def relu(z):
    return np.maximum(0, z)

H = relu(Z_hidden)
H1, H2 = H

print(f"Z1 = {Z1_manual:.4f}  ->  H1 = relu(Z1) = {H1:.4f}")
print(f"Z2 = {Z2_manual:.4f}  ->  H2 = relu(Z2) = {H2:.4f}")

Z1 = -0.7000  ->  H1 = relu(Z1) = 0.0000
Z2 = 0.1000  ->  H2 = relu(Z2) = 0.1000


### Step 3 — Output layer weighted sum ($Z_3$)

Same idea as before, but now the "inputs" to this layer are the hidden activations $H_1, H_2$ instead of $x$.

$$Z_3 = (w_{21} \cdot H_1) + (w_{22} \cdot H_2) + \theta_3$$

In [4]:
w21, w22 = W_o

Z3_manual = (w21 * H1) + (w22 * H2) + theta_o
Z3_vectorized = H @ W_o + theta_o

print(f"Z3 (manual)     = {Z3_manual}")
print(f"Z3 (vectorized) = {Z3_vectorized}")
assert np.isclose(Z3_manual, Z3_vectorized)

Z3 (manual)     = 0.08
Z3 (vectorized) = 0.08


### Step 4 — Apply the activation function to get $\hat{y}$

Since the problem states the activation as a single general rule $f = \max(0, Z_n)$ (i.e. it isn't restricted to the hidden layer only), I apply ReLU at the output as well.

In [5]:
y_hat = relu(Z3_vectorized)
print(f"Z3    = {Z3_vectorized:.4f}")
print(f"y_hat = relu(Z3) = {y_hat:.4f}")

Z3    = 0.0800
y_hat = relu(Z3) = 0.0800


### Step 5 — Compute the error

"Error" can mean a few closely related things, so I compute the common ones and explain what each is used for:

| Quantity | Formula | Meaning |
|---|---|---|
| Raw error / residual | $e = y - \hat{y}$ | Signed difference, used for reporting |
| Output error signal | $\delta = \hat{y} - y$ | The sign convention used later for backpropagation (Lab 3) |
| Absolute error | $\lvert e \rvert$ | Magnitude only |
| Squared error | $e^2 = (y-\hat{y})^2$ | Penalizes larger errors more; base of MSE |
| Half squared error (loss) | $E = \tfrac{1}{2}(y-\hat{y})^2$ | Common loss form — the $\tfrac12$ cancels nicely when differentiated |

I report all of them so the notion of "the error" is unambiguous regardless of which definition the grader expects.

In [6]:
raw_error = y[0] - y_hat            # e = y - y_hat
delta = y_hat - y[0]                # error signal convention used for backprop
abs_error = np.abs(raw_error)
squared_error = raw_error ** 2
half_squared_error = 0.5 * squared_error   # this is the "loss" E we will reuse in Lab 3

print(f"Predicted output y_hat        = {y_hat:.4f}")
print(f"Actual target y                = {y[0]:.4f}")
print("-" * 50)
print(f"Raw error        (y - y_hat)   = {raw_error:.4f}")
print(f"Error signal      (y_hat - y)  = {delta:.4f}")
print(f"Absolute error   |y - y_hat|   = {abs_error:.4f}")
print(f"Squared error    (y - y_hat)^2 = {squared_error:.4f}")
print(f"Loss E = 1/2 (y - y_hat)^2     = {half_squared_error:.4f}")

Predicted output y_hat        = 0.0800
Actual target y                = 1.0000
--------------------------------------------------
Raw error        (y - y_hat)   = 0.9200
Error signal      (y_hat - y)  = -0.9200
Absolute error   |y - y_hat|   = 0.9200
Squared error    (y - y_hat)^2 = 0.8464
Loss E = 1/2 (y - y_hat)^2     = 0.4232


### Step 6 — Vectorized, reusable implementation (sanity check)

To make sure my step-by-step math generalizes and isn't a coincidence of how I wrote the loops, I wrap the whole forward pass into one small function using proper matrix notation:

$$Z_h = xW_h + \theta_h, \qquad H = f(Z_h), \qquad Z_o = HW_o + \theta_o, \qquad \hat{y} = f(Z_o)$$

and confirm it reproduces the exact same numbers as above.

In [7]:
# Generic single-hidden-layer forward pass. Returns every intermediate
# value (Z_h, H, Z_o, y_hat) because we will reuse them for backprop in Lab 3.
def forward_pass(x, W_h, theta_h, W_o, theta_o, activation=relu):
    Z_h = x @ W_h + theta_h
    H = activation(Z_h)
    Z_o = H @ W_o + theta_o
    y_hat = activation(Z_o)
    return {"Z_h": Z_h, "H": H, "Z_o": Z_o, "y_hat": y_hat}


result = forward_pass(x, W_h, theta_h, W_o, theta_o)
for k, v in result.items():
    print(f"{k:6s} = {v}")

assert np.isclose(result["y_hat"], y_hat)
print("\nFunction output matches the manual, step-by-step result.")

Z_h    = [-0.7  0.1]
H      = [0.  0.1]
Z_o    = 0.08
y_hat  = 0.08

Function output matches the manual, step-by-step result.


### Step 7 — Cross-check with PyTorch

As a final sanity check, I rebuild the exact same tiny network using `torch.nn.Linear` layers and manually load in the given weights (instead of letting PyTorch randomly initialize them). If PyTorch's forward pass produces the same $\hat{y}$ and error as my NumPy code, I can be confident the computation is correct — and it's also a nice first look at how these formulas map onto real deep learning code before Lab 4.

Note: `nn.Linear(in_features, out_features).weight` has shape `(out_features, in_features)`, which is the **transpose** of the `W_h`/`W_o` convention I used above, so I transpose when loading the weights.

In [8]:
import torch
import torch.nn as nn

torch.manual_seed(0)

x_t = torch.tensor([[1.0, 0.0, 1.0]])        # shape (1, 3): batch of 1 sample, 3 features
y_t = torch.tensor([[1.0]])                  # shape (1, 1)

hidden_layer = nn.Linear(in_features=3, out_features=2)
output_layer = nn.Linear(in_features=2, out_features=1)

with torch.no_grad():
    # nn.Linear.weight shape = (out_features, in_features) -> transpose of W_h
    hidden_layer.weight.copy_(torch.tensor(W_h.T))
    hidden_layer.bias.copy_(torch.tensor(theta_h))
    output_layer.weight.copy_(torch.tensor(W_o).reshape(1, 2))
    output_layer.bias.copy_(torch.tensor([theta_o]))

relu_torch = nn.ReLU()

Z_h_t = hidden_layer(x_t)
H_t = relu_torch(Z_h_t)
Z_o_t = output_layer(H_t)
y_hat_t = relu_torch(Z_o_t)

loss_fn = nn.MSELoss()               # nn.MSELoss computes mean((y_hat - y)^2), no 1/2 factor
mse_t = loss_fn(y_hat_t, y_t)
half_se_t = 0.5 * (y_hat_t - y_t) ** 2

print("PyTorch Z_hidden :", Z_h_t.detach().numpy().ravel())
print("PyTorch H        :", H_t.detach().numpy().ravel())
print("PyTorch Z_output :", Z_o_t.item())
print("PyTorch y_hat    :", y_hat_t.item())
print("PyTorch MSELoss  :", mse_t.item())
print("PyTorch 1/2 SE   :", half_se_t.item())

assert np.isclose(y_hat_t.item(), y_hat, atol=1e-6)
assert np.isclose(half_se_t.item(), half_squared_error, atol=1e-6)
print("\nPyTorch forward pass matches the NumPy implementation.")

PyTorch Z_hidden : [-0.70000005  0.09999999]
PyTorch H        : [0.         0.09999999]
PyTorch Z_output : 0.07999999821186066
PyTorch y_hat    : 0.07999999821186066
PyTorch MSELoss  : 0.8464000225067139
PyTorch 1/2 SE   : 0.42320001125335693

PyTorch forward pass matches the NumPy implementation.


### Summary

- Feeding $x=[1,0,1]$ through the given hidden weights produced $Z_1 = -0.70$ and $Z_2 = 0.10$. Because $Z_1 < 0$, ReLU zeroes it out ($H_1 = 0$), while $Z_2 > 0$ passes through unchanged ($H_2 = Z_2$).
- The output pre-activation $Z_3$ ended up small and positive, so ReLU again passes it through unchanged, giving a very small prediction $\hat{y} \approx 0.08$.
- Since the target is $y = 1$, the network is very wrong at this (randomly-chosen) set of weights — the loss $E = \tfrac12(y-\hat{y})^2 \approx 0.42$ is large relative to the scale of the problem.
- This is expected: the weights given were not trained, they're just an example to illustrate the forward computation. This large error is exactly what **backpropagation** (Lab 3) is designed to reduce, by nudging every weight in the direction that decreases $E$.
- The fact that $H_1 = 0$ is worth remembering — in Lab 3 we'll see that it means **no gradient at all reaches $w_{11}, w_{13}, w_{15}, \theta_1$** on this particular update step, because their contribution to the output was already zero.